<a href="https://colab.research.google.com/github/mohamadfaisalbashir/grokking-Deep-Learning/blob/main/05_learning_multiple_weights_at_a_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Chapter 5: Learning Multiple Weights at a Time - Generalizing Gradient Descent**

This notebook summarizes Chapter 5 of *Grokking Deep Learning*:

1. Gradient Descent Learning with Multiple Inputs
2. Gradient Descent with Multiple Inputs Explained
3. Watching Several Steps of Learning
4. Freezing One Weight: What Does It Do?
5. Gradient Descent Learning with Multiple Outputs
6. Gradient Descent with Multiple Inputs and Outputs
7. What Do These Weights Learn?
8. Visualizing Weight Values
9. Visualizing Dot Products (Weighted Sums)
10. Summary

Chapter 4 showed how gradient descent updates a **single weight**. This chapter generalizes the idea to networks with **many weights**: multiple inputs, multiple outputs, and both at once. It then introduces the first real-world dataset of the book (MNIST) and offers two ways of interpreting what a network's weights actually learn.

The chapter opens with a remark attributed to Richard Branson: one learns to walk not by following rules, but by doing and by falling over.

---
## **1. Gradient Descent Learning with Multiple Inputs**

**Gradient descent also works with multiple inputs.**

Chapter 4 explained how to use gradient descent to update one weight. This chapter reveals that the same technique can update a network that contains **multiple weights**. The author begins by "jumping in the deep end": a network with three inputs and one output, whose learning is shown in four stages.

### **1.1 Stages 1 and 2: An Empty Network, Predict, and Compare**

1. **An empty network with multiple inputs.** This is the three-input network from Chapter 3. The function `w_sum` computes the weighted sum of the inputs, and the weights are `[0.1, 0.2, -.1]`, one for each of `toes`, `wlrec` (the win/loss record), and `nfans`. Note that the third weight is now `-.1` instead of `0`.
2. **Predict and compare.** The input is the first game of the season, `[toes[0], wlrec[0], nfans[0]]`, that is, 8.5, 65%, and 1.2. The target is `true = win_or_lose_binary[0]`, which is 1 (a win). The network predicts **0.86**, so `error = (pred - true) ** 2` is about **0.02**, and `delta = pred - true` is about **-0.14**: the prediction was too low by 0.14.

> An empty network with multiple inputs, and predicting, comparing, and calculating delta (stages 1 and 2)
>
> <img src="pictures/chapter 5/picture 5.1.png" alt="Figure 5.1" width="600"/>


### **1.2 Stages 3 and 4: Calculate Each `weight_delta` and Update the Weights**

3. **Learn: calculate each `weight_delta`.** The single `delta` of the output node is multiplied by each weight's own input: `weight_deltas = ele_mul(delta, input)`. The three results are approximately **-1.19**, **-0.091**, and **-0.168**. The `ele_mul` function from before can be reused, because every value is multiplied by the same number (`delta`).
4. **Learn: update the weights.** With `alpha = 0.01`, each weight moves by `weights[i] -= alpha * weight_deltas[i]`. The weights become approximately **0.1119**, **0.2009**, and **-0.098**.

> Calculating each weight_delta and updating the weights (stages 3 and 4)
>
> <img src="pictures/chapter 5/picture 5.2.png" alt="Figure 5.2" width="600"/>

### **1.3 All Four Stages in Code**

The following cell combines the code of the four stages. It should print weights close to `[0.1119, 0.2009, -0.0983]` and weight deltas close to `[-1.19, -0.091, -0.168]` (floating-point arithmetic may display more decimal places).

*Note:* one box in the book's figure for stage 3 calls `neural_network(input,weight)`. The variable of this network is `weights` (plural), which is what the other boxes use, so the cell below uses `weights`.

In [1]:
def w_sum(a,b):
    assert(len(a) == len(b))
    output = 0
    for i in range(len(a)):
        output += (a[i] * b[i])
    return output

weights = [0.1, 0.2, -.1]

def neural_network(input, weights):
    pred = w_sum(input,weights)
    return pred

toes = [8.5 , 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2 , 1.3, 0.5, 1.0]

win_or_lose_binary = [1, 1, 0, 1]
true = win_or_lose_binary[0]

input = [toes[0],wlrec[0],nfans[0]]
pred = neural_network(input,weights)

error = (pred - true) ** 2
delta = pred - true

def ele_mul(number,vector):
    output = [0,0,0]
    assert(len(output) == len(vector))
    for i in range(len(vector)):
        output[i] = number * vector[i]
    return output

weight_deltas = ele_mul(delta,input)

alpha = 0.01

for i in range(len(weights)):
    weights[i] -= alpha * weight_deltas[i]
print("Weights:" + str(weights))
print("Weight Deltas:" + str(weight_deltas))

Weights:[0.1119, 0.20091, -0.09832]
Weight Deltas:[-1.189999999999999, -0.09099999999999994, -0.16799999999999987]


---
## **2. Gradient Descent with Multiple Inputs Explained**

**Simple to execute, and fascinating to understand.**

Put side by side with the single-weight network, gradient descent with multiple inputs seems rather obvious in practice. The properties involved, however, are worth discussing.

### **2.1 Single Input versus Multiple Inputs**

> Single input and multi-input: making a prediction and calculating error and delta
>
> <img src="pictures/chapter 5/picture 5.3.png" alt="Figure 5.3" width="600"/>

Up to the generation of `delta` on the output node, the single-input and multi-input versions are **identical** (apart from the differences in how predictions are made, which were studied in Chapter 3). In both cases a prediction is made, and `error` and `delta` are calculated in the same way.

The following problem remains. With only one weight, there was only one `input` and therefore only one `weight_delta` to generate. Now there are three weights. **How can three `weight_delta` values be generated?**

### **2.2 How to Turn a Single `delta` (on the Node) into Three `weight_delta` Values**

Recall the definition and purpose of `delta` versus `weight_delta`:

> **`delta`**: a measure of how much higher or lower a node's value should be, to predict perfectly given the current training example.
>
> **`weight_delta`**: a derivative-based estimate of the direction and amount a weight should move to reduce `node_delta`, accounting for scaling, negative reversal, and stopping.

`delta` is computed by a direct subtraction between the node's value and what it should have been (`pred - true`). A positive `delta` means the node's value was too high, and a negative one means it was too low. `weight_delta`, on the other hand, is an *estimate* of the direction and amount in which to move the weights, inferred by the derivative. To transform `delta` into a `weight_delta`, **multiply `delta` by the weight's `input`**.

### **2.3 The Perspective of a Single Weight**

> One weight's perspective: the shared delta and its own input
>
> <img src="pictures/chapter 5/picture 5.4.png" alt="Figure 5.4" width="600"/>

Consider the situation from the point of view of one weight, as an imagined conversation:

- **`delta`** tells all of its inputs: "Next time, predict a little higher."
- **The single weight** reasons about its own `input`:
  - If my `input` was 0, my weight would not have mattered, and I would not change a thing (**stopping**).
  - If my `input` was negative, I would want to decrease my weight instead of increasing it (**negative reversal**).
  - But my `input` is positive and quite large, so I *guess* that my personal prediction mattered a lot to the aggregated output. I am going to move my weight up a lot to compensate (**scaling**).

All three observations (stopping, negative reversal, and scaling) are about how the weight's role in `delta` was affected by its `input`. Each `weight_delta` is therefore a sort of **input-modified version of `delta`**. Because every weight has its own `input` and all of them share the same `delta`, each `weight_delta` is obtained by multiplying that weight's `input` by `delta`.

### **2.4 Calculating `weight_delta`**

> Single input and multi-input: calculating weight_delta and putting it on the weight(s)
>
> <img src="pictures/chapter 5/picture 5.5.png" alt="Figure 5.5" width="600"/>

The two diagrams show the generation of `weight_delta` for the earlier single-input architecture and for the new multi-input architecture. The easiest way to see how similar they are is to read the pseudocode at the bottom of each box. The multi-weight version multiplies `delta` (-0.14) by **every** input to create the various `weight_deltas`. It is a simple process.

### **2.5 Updating the Weights**

> Updating the weight (single input) and updating the weights (multiple inputs)
>
> <img src="pictures/chapter 5/picture 5.6.png" alt="Figure 5.6" width="600"/>

The weight update is also nearly identical to that of the single-input network. Each `weight_delta` is multiplied by the small number `alpha` before being used to update the weight. This allows control over how quickly the network learns: if it learns too quickly, it can update the weights too aggressively and overshoot. Once the `weight_delta` values are available, they are multiplied by `alpha` and subtracted from the weights. It is literally **the same process as before, repeated across multiple weights instead of a single one**.

---
## **3. Watching Several Steps of Learning**

**Does the network keep reducing the error over several iterations?**

The listing below runs the learning loop for **three iterations**. In each iteration it predicts, computes `error` and `delta`, generates the `weight_deltas`, prints the state of the network, and finally updates the weights. It defines its own compact versions of `neural_network` and `ele_mul`, so the cell can be run on its own.

Approximately, the prediction moves from 0.86 (iteration 1) to 0.964 (iteration 2) and 0.991 (iteration 3), and the error falls from about 0.0196 to 0.0013 and then to about 0.00009.

In [2]:
def neural_network(input, weights):
    out = 0
    for i in range(len(input)):
        out += (input[i] * weights[i])
    return out

def ele_mul(scalar, vector):
    out = [0,0,0]
    for i in range(len(out)):
        out[i] = vector[i] * scalar
    return out

toes = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

win_or_lose_binary = [1, 1, 0, 1]
true = win_or_lose_binary[0]

alpha = 0.01
weights = [0.1, 0.2, -.1]
input = [toes[0],wlrec[0],nfans[0]]

for iter in range(3):

    pred = neural_network(input,weights)

    error = (pred - true) ** 2
    delta = pred - true

    weight_deltas=ele_mul(delta,input)

    print("Iteration:" + str(iter+1))
    print("Pred:" + str(pred))
    print("Error:" + str(error))
    print("Delta:" + str(delta))
    print("Weights:" + str(weights))
    print("Weight_Deltas:")
    print(str(weight_deltas))
    print(
    )

    for i in range(len(weights)):
        weights[i]-=alpha*weight_deltas[i]

Iteration:1
Pred:0.8600000000000001
Error:0.01959999999999997
Delta:-0.1399999999999999
Weights:[0.1, 0.2, -0.1]
Weight_Deltas:
[-1.189999999999999, -0.09099999999999994, -0.16799999999999987]

Iteration:2
Pred:0.9637574999999999
Error:0.0013135188062500048
Delta:-0.036242500000000066
Weights:[0.1119, 0.20091, -0.09832]
Weight_Deltas:
[-0.30806125000000056, -0.023557625000000044, -0.04349100000000008]

Iteration:3
Pred:0.9906177228125002
Error:8.802712522307997e-05
Delta:-0.009382277187499843
Weights:[0.11498061250000001, 0.20114557625, -0.09788509000000001]
Weight_Deltas:
[-0.07974935609374867, -0.006098480171874899, -0.011258732624999811]



### **3.1 One Error Curve per Weight**

> Several steps of learning: iteration 1, with one error/weight curve for each weight (a, b, and c)
>
> <img src="pictures/chapter 5/picture 5.7.png" alt="Figure 5.7" width="600"/>

As before, an individual error/weight curve can be drawn for each weight. The slopes of these curves (the dotted lines) are reflected by the `weight_delta` values. Notice that **a** (the weight of the first input, 8.5) is steeper than the others. Why is its `weight_delta` steeper if the weights share the same output `delta` and `error` measure? Because **a** has an `input` value that is significantly higher than the others, and thus a higher derivative.

> Several steps of learning: iterations 2 and 3
>
> <img src="pictures/chapter 5/picture 5.8.png" alt="Figure 5.8" width="600"/>

### **3.2 Additional Takeaways**

- Most of the learning (weight changing) was performed on the weight with the **largest input** (**a**), because the input changes the slope significantly.
- This is not necessarily advantageous in all settings. A subfield called **normalization** helps encourage learning across all weights despite dataset characteristics such as this one.
- The significant difference in slope forced the author to set `alpha` **lower than wanted** (0.01 instead of 0.1). As an exercise, set `alpha` to 0.1 in the cell above and observe how **a** causes the network to diverge.

---
## **4. Freezing One Weight: What Does It Do?**

**An experiment to understand how the weights affect each other.**

This experiment is a bit advanced in terms of theory, but it is a great exercise for understanding how the weights influence one another. The network is trained again, except that weight **a** (`weights[0]`) is **never adjusted**. The training example has to be learned using only weights **b** and **c** (`weights[1]` and `weights[2]`).

The code differs from the previous listing in two places (shown in bold in the book): the line `weight_deltas[0] = 0`, which cancels the update of the first weight, and `alpha = 0.3`.

In [3]:
def neural_network(input, weights):
    out = 0
    for i in range(len(input)):
        out += (input[i] * weights[i])
    return out

def ele_mul(scalar, vector):
    out = [0,0,0]
    for i in range(len(out)):
        out[i] = vector[i] * scalar
    return out

toes = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

win_or_lose_binary = [1, 1, 0, 1]
true = win_or_lose_binary[0]

alpha = 0.3
weights = [0.1, 0.2, -.1]
input = [toes[0],wlrec[0],nfans[0]]

for iter in range(3):

    pred = neural_network(input,weights)

    error = (pred - true) ** 2
    delta = pred - true

    weight_deltas=ele_mul(delta,input)
    weight_deltas[0] = 0

    print("Iteration:" + str(iter+1))
    print("Pred:" + str(pred))
    print("Error:" + str(error))
    print("Delta:" + str(delta))
    print("Weights:" + str(weights))
    print("Weight_Deltas:")
    print(str(weight_deltas))
    print(
    )

    for i in range(len(weights)):
        weights[i]-=alpha*weight_deltas[i]

Iteration:1
Pred:0.8600000000000001
Error:0.01959999999999997
Delta:-0.1399999999999999
Weights:[0.1, 0.2, -0.1]
Weight_Deltas:
[0, -0.09099999999999994, -0.16799999999999987]

Iteration:2
Pred:0.9382250000000001
Error:0.003816150624999989
Delta:-0.06177499999999991
Weights:[0.1, 0.2273, -0.04960000000000005]
Weight_Deltas:
[0, -0.040153749999999946, -0.07412999999999989]

Iteration:3
Pred:0.97274178125
Error:0.000743010489422852
Delta:-0.027258218750000007
Weights:[0.1, 0.239346125, -0.02736100000000008]
Weight_Deltas:
[0, -0.017717842187500006, -0.032709862500000006]



> Freezing one weight: iteration 1
>
> <img src="pictures/chapter 5/picture 5.9.png" alt="Figure 5.9" width="600"/>

> Freezing one weight: iterations 2 and 3
>
> <img src="pictures/chapter 5/picture 5.10.png" alt="Figure 5.10" width="600"/>

Running the cell, the prediction moves from 0.86 to about 0.938 and then to about 0.973, while the error falls from about 0.0196 to 0.0038 and then to 0.0007, even though `weights[0]` stays at 0.1.

### **4.1 Observation: Weight a Still Finds the Bottom of the Bowl**

It may be surprising that **a** still finds the bottom of its bowl. The reason is that the curves measure each individual weight relative to the **global error**. Because `error` is shared, **when one weight finds the bottom of the bowl, all the weights find the bottom of the bowl**.

### **4.2 Lesson 1: A Potentially Damaging Property**

Suppose the **b** and **c** weights had converged (reached `error = 0`) and only then **a** were trained. Weight **a** would not move: `error = 0` implies that its `weight_delta` is 0.

This reveals a potentially damaging property of neural networks. Weight **a** may belong to a powerful input with a lot of predictive power, but **if the network accidentally figures out how to predict accurately on the training data without it, it will never learn to incorporate that input into its prediction**.

### **4.3 Lesson 2: The Error Plane**

Notice also how **a** finds the bottom of the bowl. Instead of the black dot moving, the curve seems to move to the left. The black dot can move horizontally only if the weight is updated, and the weight of **a** is frozen in this experiment, so the dot must stay fixed. Nevertheless, `error` clearly goes to 0.

This tells us what the graphs really are. In truth, they are **2D slices of a four-dimensional shape**: three of the dimensions are the weight values, and the fourth dimension is the error. This shape is called the **error plane**, and, believe it or not, its curvature is determined by the **training data**. The `error` is determined by the training data: any network can have any `weight` value, but the value of `error` for any particular weight configuration is 100% determined by the data. What the network is really trying to do is find the **lowest point on this big error plane**, where the lowest point means the lowest `error`. The author notes that this idea will return later.

---
## **5. Gradient Descent Learning with Multiple Outputs**

**Neural networks can also make multiple predictions using only a single input.**

This case may seem a bit obvious. Each `delta` is calculated in the same way, and then each of them is multiplied by the same, single input. This becomes the `weight_delta` of each weight. At this point it should be clear that one simple mechanism (**stochastic gradient descent**) is used consistently to perform learning across a wide variety of architectures.

The four stages are:

1. **An empty network with multiple outputs.** One input (the win/loss record) feeds three outputs (`hurt?`, `win?`, `sad?`) through the weights `[0.3, 0.2, 0.9]`. The network predicts whether the team won and also whether the players are happy or sad and what percentage of them are hurt, using only the current win/loss record.
2. **Predict and calculate `error` and `delta` for every output.** With the first game (`wlrec[0] = 0.65`), the predictions are approximately `[0.195, 0.13, 0.585]` and the targets are `[hurt[0], win[0], sad[0]] = [0.1, 1, 0.1]`. The `error` and `delta` of each output are calculated with a loop, giving deltas of approximately `[0.095, -0.87, 0.485]` and errors of approximately `[0.009, 0.757, 0.235]`.
3. **Compare: calculate each `weight_delta`.** As before, each `weight_delta` is the input node value multiplied by the `delta` of the output node it connects to. Here, the weights **share the same input node** and have **unique output nodes**. The `ele_mul` function can be reused. The `weight_deltas` are approximately `[0.062, -0.566, 0.315]`.
4. **Learn: update the weights.** With `alpha = 0.1`, each weight is updated with `weights[i] -= (weight_deltas[i] * alpha)`. The weights become approximately `[0.294, 0.257, 0.868]`.

> Multiple outputs: an empty network, and predicting and calculating error and delta (stages 1 and 2)
>
> <img src="pictures/chapter 5/picture 5.11.png" alt="Figure 5.11" width="600"/>

> Multiple outputs: calculating each weight_delta and updating the weights (stages 3 and 4)
>
> <img src="pictures/chapter 5/picture 5.12.png" alt="Figure 5.12" width="600"/>

*Notes on the code below:*

- The book's `neural_network` calls `ele_mul`, which was defined earlier in the chapter (in the code of Section 1). It is included at the top of the cell so that the cell can be run on its own. The function `scalar_ele_mul` of this stage has the same body.
- In the book's listing, the line that creates the `weight_deltas` passes `weights` as its second argument. The figure and the accompanying text (the input node value multiplied by the output node `delta` of each weight) correspond to passing `delta`, which gives the weights shown in the figure (about .29, .26, and .87). The cell below therefore uses `scalar_ele_mul(input,delta)`.
- In the figure, the circles for `error` and `delta` of the second and third outputs appear interchanged. The values above are the ones given by the formulas.

In [4]:
def ele_mul(number,vector):
    output = [0,0,0]
    assert(len(output) == len(vector))
    for i in range(len(vector)):
        output[i] = number * vector[i]
    return output

weights = [0.3, 0.2, 0.9]

def neural_network(input, weights):
    pred = ele_mul(input,weights)
    return pred

wlrec = [0.65, 1.0, 1.0, 0.9]

hurt  = [0.1, 0.0, 0.0, 0.1]
win   = [  1,   1,   0,   1]
sad   = [0.1, 0.0, 0.1, 0.2]

input = wlrec[0]
true = [hurt[0], win[0], sad[0]]

pred = neural_network(input,weights)

error = [0, 0, 0]
delta = [0, 0, 0]

for i in range(len(true)):

    error[i] = (pred[i] - true[i]) ** 2
    delta[i] = pred[i] - true[i]

def scalar_ele_mul(number,vector):

    output = [0,0,0]

    assert(len(output) == len(vector))

    for i in range(len(vector)):
        output[i] = number * vector[i]

    return output

weight_deltas = scalar_ele_mul(input,delta)
alpha = 0.1

for i in range(len(weights)):
    weights[i] -= (weight_deltas[i] * alpha)

print("Weights:" + str(weights))
print("Weight Deltas:" + str(weight_deltas))

Weights:[0.293825, 0.25655, 0.868475]
Weight Deltas:[0.061750000000000006, -0.5655, 0.3152500000000001]


---
## **6. Gradient Descent with Multiple Inputs and Outputs**

**Gradient descent generalizes to arbitrarily large networks.**

The last architecture combines the two previous ones: **every input is connected to every output** through its own weight, so the weights form a matrix (3 x 3 in this example), as in the network of Chapter 3.

1. **An empty network with multiple inputs and outputs.** The weights are `[[0.1, 0.1, -0.3], [0.1, 0.2, 0.0], [0.0, 1.3, 0.1]]` (rows for `hurt?`, `win?`, and `sad?`; columns for `toes`, `win`, and `fans`). The function `vect_mat_mul` computes one weighted sum per output.
2. **Predict and calculate `error` and `delta` for every output.** For the first game, the predictions are approximately `[0.555, 0.98, 0.965]`, the deltas are approximately `[0.455, -0.02, 0.865]`, and the errors are approximately `[0.207, 0.0004, 0.748]`.
3. **Compare: calculate each `weight_delta`.** There is now one `weight_delta` for every weight, obtained by multiplying every input by every output `delta`. This operation is called an **outer product** and is implemented by the function `outer_prod`. The figure shows the `weight_deltas` of only one input (the win/loss record) to save space.
4. **Learn: update the weights.** Every weight in the matrix is updated with `weights[i][j] -= alpha * weight_deltas[i][j]`, using `alpha = 0.01`.

> Multiple inputs and outputs: an empty network, and predicting and calculating error and delta (stages 1 and 2)
>
> <img src="pictures/chapter 5/picture 5.13.png" alt="Figure 5.13" width="600"/>

> Multiple inputs and outputs: calculating each weight_delta and updating the weights (stages 3 and 4)
>
> <img src="pictures/chapter 5/picture 5.14.png" alt="Figure 5.14" width="600"/>

*Notes on the code below:*

- The book's `outer_prod` creates its result with `zeros_matrix(len(a),len(b))` and loops over `a` and `b`, but neither `zeros_matrix` nor `a` and `b` are defined (the parameters are `vec_a` and `vec_b`). In the cell below, the matrix is created with NumPy (`np.zeros((len(vec_a),len(vec_b)))`, as in the NumPy primer of Chapter 3, with `import numpy as np` added) and the loops use `vec_a` and `vec_b`.
- `w_sum` (from stage 1 of Section 1) is included, because `vect_mat_mul` depends on it.
- In this listing `weights[i]` holds the weights that feed output `i`, whereas `outer_prod(input,delta)` returns a matrix indexed as `[input][output]`. The update loop pairs the two index by index, so they are in opposite orientations; pairing `weights[i][j]` with `weight_deltas[j][i]` would match them. The cell still runs because the matrix is square, and it follows the book's listing.
- In the figure, the second error is printed as .96, while the formula gives about 0.0004.

In [5]:
import numpy as np

def w_sum(a,b):
    assert(len(a) == len(b))
    output = 0
    for i in range(len(a)):
        output += (a[i] * b[i])
    return output

            # toes %win # fans
weights = [ [0.1, 0.1, -0.3],# hurt?
            [0.1, 0.2, 0.0], # win?
            [0.0, 1.3, 0.1] ]# sad?

def vect_mat_mul(vect,matrix):
    assert(len(vect) == len(matrix))
    output = [0,0,0]
    for i in range(len(vect)):
        output[i] = w_sum(vect,matrix[i])
    return output

def neural_network(input, weights):

    pred = vect_mat_mul(input,weights)

    return pred

toes =  [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65,0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

hurt  = [0.1, 0.0, 0.0, 0.1]
win   = [  1,   1,   0,   1]
sad   = [0.1, 0.0, 0.1, 0.2]

alpha = 0.01

input = [toes[0],wlrec[0],nfans[0]]
true = [hurt[0], win[0], sad[0]]

pred = neural_network(input,weights)

error = [0, 0, 0]
delta = [0, 0, 0]

for i in range(len(true)):

    error[i] = (pred[i] - true[i]) ** 2
    delta[i] = pred[i] - true[i]

def outer_prod(vec_a, vec_b):

    out = np.zeros((len(vec_a),len(vec_b)))

    for i in range(len(vec_a)):
        for j in range(len(vec_b)):
            out[i][j] = vec_a[i]*vec_b[j]

    return out

weight_deltas = outer_prod(input,delta)

for i in range(len(weights)):
  for j in range(len(weights[0])):
    weights[i][j] -= alpha * \
                     weight_deltas[i][j]

---
## **7. What Do These Weights Learn?**

**Each weight tries to reduce the error, but what do they learn in aggregate?**

### **7.1 The MNIST Dataset**

This is the part of the book where the first **real-world dataset** appears, and it is one with historical significance. The **MNIST** dataset (Modified National Institute of Standards and Technology) consists of digits that high school students and employees of the US Census Bureau handwrote some years ago. The digits are **black-and-white images** of handwriting, and each digit image is accompanied by the actual number it represents (0 to 9). For the last few decades, people have used this dataset to train neural networks to read human handwriting, and now the reader does the same.

> A handwritten digit from the MNIST dataset
>
> <img src="pictures/chapter 5/picture 5.15.png" alt="Figure 5.15" width="600"/>

### **7.2 The Shape of the Network**

Each image has only **784 pixels** (28 x 28). With 784 pixels as input and 10 possible labels as output, the shape of the network follows directly:

- Every training example contains 784 values (one per pixel), so the network must have **784 input nodes**. The number of input nodes is adjusted to reflect how many datapoints each training example has.
- The network must predict **10 probabilities**, one for each digit. Given an input drawing, it produces these 10 probabilities, which tell which digit is most likely to have been drawn.

This is the multiple-input, multiple-output network from the previous section, adapted to 784 inputs and 10 outputs.

The book's accompanying **MNISTPreprocessor** notebook contains a script that loads the first 1,000 images and labels into two NumPy matrices called `images` and `labels`. Because images are two-dimensional (28 x 28) but the network is flat, each image is **flattened into a vector of 1 x 784**: the first row of pixels, followed by the second row, then the third, and so on, until there is one list of 784 pixels per image. (This preprocessing code is not printed in the chapter, so it is not reproduced here.)

> The MNIST classification network: 784 inputs (one per pixel) and 10 outputs (one per digit)
>
> <img src="pictures/chapter 5/picture 5.16.png" alt="Figure 5.16" width="600"/>

This network most closely resembles the multiple-input, multiple-output network trained earlier. The only difference is the number of inputs and outputs, which has increased substantially. If it could predict perfectly, it would take in an image's pixels and predict a **1.0 in the correct output position and a 0 everywhere else**. If it did this correctly for all images in the dataset, it would have no error.

> A prediction for an image of a 2: the highest output is the correct one
>
> <img src="pictures/chapter 5/picture 5.17.png" alt="Figure 5.17" width="600"/>

In the example, the output for the digit 2 is the highest prediction (0.98), so the network thinks the image is a 2. The other outputs are small errors. For instance, the network thinks the image kind of looks like a 9, but only a bit (0.15).

Over the course of training, the network adjusts the weights between the input and prediction nodes so that `error` falls toward 0 on the training data. The question is what this actually does: **what does it mean to modify a bunch of weights to learn a pattern in aggregate?** The next two sections answer it.

---
## **8. Visualizing Weight Values**

An interesting and intuitive practice in neural network research (particularly for image classifiers) is to **visualize the weights as if they were an image**.

Each output node has one weight coming from every pixel. For example, the `2?` node has 784 input weights, each mapping the relationship between a pixel and the number 2. What does that relationship mean?

- If a weight is **high**, the model believes there is a high degree of **correlation** between that pixel and the number 2.
- If it is **very low (negative)**, the network believes there is a very low correlation (perhaps even negative correlation) between that pixel and the number 2.

If the weights of an output node are printed out as an image of the same shape as the input images, it becomes possible to see which pixels have the highest correlation with that output node.

> Visualizing weight values: the weights of the 2? and 1? output nodes drawn as images
>
> <img src="pictures/chapter 5/picture 5.18.png" alt="Figure 5.18" width="600"/>

In the example, a very vague **2** and a very vague **1** appear in the two images, which were created using the weights for 2 and for 1, respectively. The **bright areas are high weights**, and the **dark areas are negative weights**. The neutral color (red, in the colored version of the book) represents 0 in the weight matrix. This illustrates that the network generally knows the shape of a 2 and of a 1.

Why does it turn out this way? This takes us back to the lesson on dot products, which is reviewed next.

---
## **9. Visualizing Dot Products (Weighted Sums)**

Recall how a dot product works: it takes two vectors, multiplies them together (elementwise), and then sums the result. Consider this example:

```
a = [ 0, 1, 0, 1]
b = [ 1, 0, 1, 0]

[ 0, 0, 0, 0] -> 0
```

Each element of `a` and `b` is first multiplied, here producing a vector of 0s. The sum of this vector is also 0 (the **score**). Why? Because the vectors have **nothing in common**.

```
c = [ 0, 1, 1, 0]
d = [.5, 0,.5, 0]
```

```
b = [ 1, 0, 1, 0]
c = [ 0, 1, 1, 0]
```

The dot products between `c` and `d`, and between `b` and `c`, return **higher scores**, because there is overlap in the columns that have positive values. Dot products between two identical vectors also tend to give higher scores. The takeaway is that **a dot product is a loose measurement of similarity between two vectors**.

### **9.1 What This Means for Weights and Inputs**

If the `weight` vector is similar to the `input` vector for a 2, it will output a **high score**, because the two vectors are similar. Conversely, if the `weight` vector is *not* similar to the `input` vector for a 2, it will output a **low score**.

> Visualizing dot products: an input image multiplied (dot) by two weight images gives two predictions
>
> <img src="pictures/chapter 5/picture 5.19.png" alt="Figure 5.19" width="600"/>

In the figure, the top score (0.98) is higher than the lower one (0.01) because the weights of the first output node resemble a 2, whereas the weights of the second output node (the 1) do not. In other words, the network learns **templates** in its weights, and a prediction is a measurement of how similar the input is to each template.

---
## **10. Summary**

**Gradient descent is a general learning algorithm.**

Perhaps the most important subtext of this chapter is that gradient descent is a very flexible learning algorithm. If weights are combined in a way that allows an error function and a `delta` to be calculated, gradient descent can show how to move the weights to reduce the error. The rest of the book explores different types of weight combinations and error functions for which gradient descent is useful, and the next chapter is no exception.

The key takeaways of Chapter 5 are:

- With **multiple inputs**, the network has a single `delta` on the output node, and each weight receives its own `weight_delta = input * delta`. The `input` provides **scaling, negative reversal, and stopping** for each weight. The update is `weights[i] -= alpha * weight_deltas[i]`.
- Each weight has its own **error/weight curve**. Weights with larger inputs have steeper curves and learn faster, which can force `alpha` to be set low. **Normalization** is a subfield that helps encourage learning across all weights.
- Because `error` is **shared**, when one weight finds the bottom of its bowl, all the weights do. If the network reaches zero error without using a particular input, the weight of that input stops moving, so the network never learns to use it. The graphs are 2D slices of a higher-dimensional **error plane** whose shape is determined by the **training data**.
- With **multiple outputs**, each output has its own `delta`, and each `weight_delta` is the shared input multiplied by the `delta` of the output it connects to.
- With **multiple inputs and outputs**, the `weight_deltas` form a matrix (the **outer product** of the input and the `delta` vector), and every weight in the matrix is updated in the same way.
- On the **MNIST** dataset, the network has **784 inputs** (one per pixel) and **10 outputs** (one per digit). When the weights of an output node are **visualized as an image**, a vague shape of the digit appears.
- This happens because a **dot product is a loose measurement of similarity**: an output is high when the input resembles the weights, so the weights act as learned templates.